# Week 2 · Test-Time Training (TTT) Layers

**Repository:** `bitwise-llm-forge` · **Theory doc:** [`docs/theory/week2_ttt.md`](../docs/theory/week2_ttt.md)

---

### Learning objectives

1. State the architectural trilemma between RNNs, full attention, and TTT — and place each in time / memory / expressivity coordinates.
2. Derive the **rank-1 inner update**  $W_t = W_{t-1} - \eta(W_{t-1}k_t - v_t)k_t^\top$  from the squared inner loss.
3. Prove the equivalence between TTT-Linear with $\eta=1$ and **linear attention with the identity feature map**.
4. Implement `TTTLinear` and verify it on a toy associative-recall task.
5. Empirically compare wall-time and peak memory of TTT vs. attention as sequence length grows.


In [ ]:
# Make ``src/`` importable when this notebook is launched from anywhere.
from pathlib import Path
import sys

_here = Path.cwd()
for candidate in (_here, *_here.parents):
    if (candidate / "src").is_dir():
        sys.path.insert(0, str(candidate))
        break

import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

from src.utils.seeding import set_seed
set_seed(42)

print("environment ready · torch", __import__("torch").__version__)


## 1 · Mathematical derivation

### 1.1 The trilemma

| Family | Time / token | Memory | State expressivity |
|---|---|---|---|
| RNN  | $O(d^2)$ | $O(d)$ | constant-rank vector |
| Attention | $O(N d)$ amortized | $O(N d)$ KV-cache | full token recall |
| **TTT-Linear** | $O(d^2)$ | $O(d^2)$ | full-rank linear function |

### 1.2 Hidden state as a learned function

TTT reinterprets the hidden state as the parameters of an inner model:

$$
W_t \in \mathbb{R}^{d\times d},\qquad f_{W_t}(z) = W_t z.
$$

### 1.3 Inner self-supervised loss

At each step the inner model is trained to reconstruct $\theta_V(x_t)$ from $\theta_K(x_t)$:

$$
\ell(W; x_t) = \tfrac{1}{2}\|W\,\theta_K(x_t) - \theta_V(x_t)\|_2^2.
$$

A single SGD step yields the **rank-1 update**

$$
W_t = W_{t-1} - \eta\big(W_{t-1}\theta_K(x_t) - \theta_V(x_t)\big)\theta_K(x_t)^\top.
$$

### 1.4 Output and linear-attention equivalence

The output at step $t$ is $y_t = W_t\,\theta_Q(x_t)$. When $\eta=1$ and $W_0=0$,

$$
W_t = \sum_{s\le t} \theta_V(x_s)\theta_K(x_s)^\top
$$

— exactly **linear attention with the identity feature map**. TTT generalizes it via tunable inner learning rate and non-linear inner models.


## 2 · Reference implementation

In [ ]:
import torch
import torch.nn as nn
import time, json
import matplotlib.pyplot as plt
from pathlib import Path

from src.ttt import TTTLinear

torch.manual_seed(0)

layer = TTTLinear(d_model=32, inner_lr=1.0)
x = torch.randn(2, 16, 32)
y = layer(x)
print("input  shape :", tuple(x.shape))
print("output shape :", tuple(y.shape))
print("number of params:", sum(p.numel() for p in layer.parameters()))


## 3 · Sanity checks

In [ ]:
# 3.1 Zero input -> zero output (W stays at zero throughout the recurrence).
x_zero = torch.zeros(1, 8, 16)
layer_small = TTTLinear(d_model=16)
y_zero = layer_small(x_zero)
print("zero-input ->  output max abs:", float(y_zero.abs().max().item()))
assert torch.allclose(y_zero, torch.zeros_like(y_zero))

# 3.2 Gradients flow through the inner SGD recurrence.
x = torch.randn(1, 8, 16, requires_grad=True)
y = layer_small(x)
y.sum().backward()
grad_norms = {name: float(p.grad.norm().item()) for name, p in layer_small.named_parameters()}
print("\ngradient L2 norms by parameter:")
for k, v in grad_norms.items():
    print(f"  {k:20s}  {v:.4f}")
assert all(v > 0 for v in grad_norms.values()), "some parameters got no gradient"
print("\nall parameters received non-zero gradients ✓")


## 4 · A toy associative-recall task

To see TTT learning something interesting, we use the **associative-recall** task often used in linear-attention papers: feed a stream of (key, value) pairs, then end with a query that matches one earlier key; the model must output the corresponding value.

This task is intentionally too small for full attention to be necessary — but it's a good unit test for the state update being information-preserving.


In [ ]:
def make_recall_batch(batch: int, n_pairs: int, d: int) -> tuple[torch.Tensor, torch.Tensor]:
    keys = torch.randn(batch, n_pairs, d)
    vals = torch.randn(batch, n_pairs, d)
    # Stream: k1 v1 k2 v2 ... kN vN  q  (where q is a copy of one of the keys)
    seq = torch.empty(batch, 2 * n_pairs + 1, d)
    seq[:, 0::2, :][:, :n_pairs, :] = keys   # k_i at even positions
    seq[:, 1::2, :] = vals                    # v_i at odd positions
    idx = torch.randint(0, n_pairs, (batch,))
    target = vals[torch.arange(batch), idx]
    seq[:, -1, :] = keys[torch.arange(batch), idx]
    return seq, target


torch.manual_seed(7)
d_model, n_pairs, batch = 16, 6, 32
model = TTTLinear(d_model=d_model, inner_lr=1.0)
opt = torch.optim.AdamW(model.parameters(), lr=3e-3)

history = []
for step in range(300):
    seq, target = make_recall_batch(batch, n_pairs, d_model)
    out = model(seq)
    pred = out[:, -1, :]            # use the last position as the query response
    loss = (pred - target).pow(2).mean()
    opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
    if step % 25 == 0:
        history.append((step, float(loss.item())))
        print(f"step {step:4d}  loss = {loss.item():.4f}")

# Plot
xs, ys = zip(*history)
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(xs, ys, marker="o"); ax.set_xlabel("step"); ax.set_ylabel("MSE loss")
ax.set_title(f"TTT-Linear learning associative recall (d={d_model}, {n_pairs} pairs)")
ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()


## 5 · Empirical benchmark — TTT vs. attention as N grows

We compare three layers on a forward pass over increasingly long sequences:

- `TTTLinear` (this repo) — $O(N d^2)$ time, $O(d^2)$ state
- `NaiveAttention` — textbook $O(N^2 d)$ time, $O(N^2)$ memory
- `F.scaled_dot_product_attention` — PyTorch fused kernel (good baseline)

Wall-time is measured as the median of 5 runs after a 2-iteration warm-up.


In [ ]:
import torch.nn.functional as F
from src.attention import NaiveAttention

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("running on", device)

d_model, n_heads = 64, 4
seq_lengths = [128, 256, 512, 1024, 2048] if device.type == "cuda" else [64, 128, 256, 512]

ttt = TTTLinear(d_model=d_model).to(device).eval()
attn = NaiveAttention(d_model, n_heads).to(device).eval()


def median_time(fn, warmup: int = 2, iters: int = 5) -> float:
    for _ in range(warmup):
        fn()
    if device.type == "cuda":
        torch.cuda.synchronize()
    samples = []
    for _ in range(iters):
        t0 = time.perf_counter()
        fn()
        if device.type == "cuda":
            torch.cuda.synchronize()
        samples.append((time.perf_counter() - t0) * 1000.0)
    samples.sort()
    return samples[len(samples) // 2]


rows = []
with torch.no_grad():
    for N in seq_lengths:
        x = torch.randn(1, N, d_model, device=device)
        t_ttt  = median_time(lambda: ttt(x))
        t_attn = median_time(lambda: attn(x))

        # Fused SDPA path
        q = torch.randn(1, n_heads, N, d_model // n_heads, device=device)
        k = torch.randn(1, n_heads, N, d_model // n_heads, device=device)
        v = torch.randn(1, n_heads, N, d_model // n_heads, device=device)
        t_sdpa = median_time(lambda: F.scaled_dot_product_attention(q, k, v, is_causal=True))

        rows.append((N, t_ttt, t_attn, t_sdpa))
        print(f"N={N:>5}  | TTT {t_ttt:7.2f} ms | naive-attn {t_attn:7.2f} ms | SDPA {t_sdpa:7.2f} ms")


# Plot
fig, ax = plt.subplots(figsize=(7.5, 4))
ns = [r[0] for r in rows]
ax.plot(ns, [r[1] for r in rows], "-o", label="TTT-Linear (O(N))")
ax.plot(ns, [r[2] for r in rows], "-s", label="Naive attention (O(N²))")
ax.plot(ns, [r[3] for r in rows], "-^", label="Fused SDPA")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("sequence length N"); ax.set_ylabel("forward latency (ms)")
ax.set_title("TTT vs. attention scaling")
ax.grid(True, which="both", alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()

# Persist
out = {"device": str(device), "rows": [
    {"N": r[0], "ttt_ms": r[1], "naive_attn_ms": r[2], "sdpa_ms": r[3]} for r in rows
]}
Path("../benchmarks/results/ttt_vs_attention.json").parent.mkdir(parents=True, exist_ok=True)
Path("../benchmarks/results/ttt_vs_attention.json").write_text(json.dumps(out, indent=2))
print("\nresults saved to ../benchmarks/results/ttt_vs_attention.json")


## 6 · Discussion

- **Linear in N.** On CPU and small-N GPU, naive attention has a small constant overhead that beats TTT's per-token Python loop. As N grows TTT's slope (linear) decisively beats attention's slope (quadratic).
- **Recurrent form is the Python bottleneck.** The reference implementation is a `for t in range(N)` Python loop. The chunked / parallel formulation (Sun et al., 2024, §3) eliminates this and is what real TTT models use.
- **TTT ≠ linear attention.** Equivalence holds *only* for `inner_lr=1`, `W_0=0`, identity feature map. Once you set `inner_lr ≠ 1` or take multiple inner SGD steps, TTT becomes a strictly larger model class.
- **Inner-model choice matters.** Replacing the linear inner model with a 2-layer MLP (TTT-MLP) gives further gains at the cost of a quadratically larger state.

### References
- Sun, Y. et al. "Learning to (Learn at Test Time): RNNs with Expressive Hidden States." arXiv:2407.04620, 2024.
- Katharopoulos, A. et al. "Transformers are RNNs: Fast Autoregressive Transformers with Linear Attention." ICML 2020.
